# GRACE H3-v2 — Bắt đầu full pipeline local

Branch: **feat/h3-v2-benchmark**. MCC là chính, F1 thứ hai; cấu hình mặc định Devign + ReVeal, seed42.

Nếu đang khôi phục validation cũ bị Vertex chặn, chạy notebooks/retry-blocked-validation.ipynb trước, rồi khởi động lại kernel riêng của entry này và chạy từ Bước 1/2. Recovery trộn Qwen/Vertex và DeepSeek/FPT; kết quả chỉ dùng exploratory.

| Lượt | Việc làm | Tiếp tục ở đâu |
|---|---|---|
| 1 | Entry tạo base-inputs local | Chạy colab_cpu_prepare.ipynb trên máy, PHASE='base' |
| 2 | Entry nhập base-output, chạy GRACE validation | Chạy lại entry; sau đó runner local, PHASE='meta' |
| 3 | Entry nhập meta-output, chạy GRACE test và đánh giá | Xem benchmark_summary.json trong runs/h3v2/ |

Notebook CPU giữ tên cũ để không mất đường dẫn, nhưng chạy hoàn toàn local: không Colab, Drive, upload hay download. Mọi zip input/output/checkpoint nằm trong bundles/.

CPU stages chạy offline khi dependencies và Java21 đã cài sẵn. Vertex LLM cần mạng trừ khi exact response cache đủ mọi request. RUN_INFERENCE=True bật gọi API; False chỉ dùng cache.

H3-v2 chưa có benchmark. Test không dùng để chọn winner.

**Profile hiện tại: validation 8.192 / test 8.192 byte UTF-8.** Snapshot kết quả test 24k đã được commit trước khi đổi profile. Việc trộn Qwen/Vertex và DeepSeek/FPT trong validation vẫn được ghi là exploratory.

**Chỉ chạy lại Devign test:** bỏ qua các bước full pipeline, chuyển thẳng đến mục “Chạy lại RIÊNG Devign test” cuối notebook và chạy R1 → R4.


In [ ]:
from pathlib import Path
import os, sys, json, subprocess
candidates=[Path.cwd(),Path.cwd().parent,Path('D:/JUNIOR/LabThayTho/GRACE-H3-BenchmarkEnsemble'),Path('/mnt/d/JUNIOR/LabThayTho/GRACE-H3-BenchmarkEnsemble')]
if os.environ.get('GRACE_H3V2_ROOT'): candidates.insert(0,Path(os.environ['GRACE_H3V2_ROOT']))
ROOT=next((p.resolve() for p in candidates if (p/'configs/benchmark_v2.json').exists()),None)
if ROOT is None: raise RuntimeError('Mở notebook từ worktree GRACE-H3-BenchmarkEnsemble hoặc đặt GRACE_H3V2_ROOT')
sys.path.insert(0,str(ROOT))
INSTALL_DEPENDENCIES=False
PREPARE_IN_LOCAL_NOTEBOOK=True
RUN_INFERENCE=True
TEST_CONTEXT_LIMIT_BYTES=8192
BUNDLES=ROOT/'bundles'
BUNDLES.mkdir(exist_ok=True)
print('ROOT:',ROOT)
print('Entry local của branch benchmark; CPU runner chạy trên cùng máy.')


In [ ]:
if INSTALL_DEPENDENCIES:
    subprocess.check_call([sys.executable,'-m','pip','install','--break-system-packages','-r',str(ROOT/'requirements-v2.txt'),'-r',str(ROOT/'requirements.txt')])
from grace_h1.io import load_local_environment
load_local_environment(ROOT)
from grace_h3v2 import workflow as w
from grace_h3v2.portable import create_bundle,import_bundle,identity
import zipfile
import pandas as pd
CFG=w.config(ROOT)
import importlib.util
_context_spec=importlib.util.spec_from_file_location('h3_test_context_profile', ROOT/'tools/test_context_profile.py')
test_context=importlib.util.module_from_spec(_context_spec)
_context_spec.loader.exec_module(test_context)
print(json.dumps(CFG,indent=2))
def show_stage(name):
    try: return w.load_stage(ROOT,CFG,name)
    except FileNotFoundError: return None
def cpu_handoff(phase,selection):
    result=show_stage(selection)
    if result is not None:
        print(f'Using local {selection}; validating existing artifacts, skipping output ZIP import.')
        return w.base_prepare(ROOT,CFG) if phase=='base' else w.meta_prepare(ROOT,CFG)
    output=BUNDLES/f'grace-h3v2-{phase}-output.zip'
    if output.exists(): print(import_bundle(ROOT,output,'output',phase))
    result=show_stage(selection)
    if result is not None: return result
    if not PREPARE_IN_LOCAL_NOTEBOOK:
        return w.base_prepare(ROOT,CFG) if phase=='base' else w.meta_prepare(ROOT,CFG)
    archive=BUNDLES/f'grace-h3v2-{phase}-inputs.zip'
    reuse=False
    if archive.exists():
        with zipfile.ZipFile(archive) as z:
            m=json.loads(z.read('bundle_manifest.json'))
            reuse=m.get('job')==identity(ROOT) and m.get('phase')==phase and m.get('kind')=='inputs'
    if not reuse: print(create_bundle(ROOT,archive,'inputs',phase))
    raise RuntimeError(f'DỪNG: mở notebooks/colab_cpu_prepare.ipynb trên máy local, đặt PHASE={phase!r} và Run All. Input: {archive}. Output sẽ ghi vào {output}. Sau đó quay lại notebook này và Run All.')


## Điền trực tiếp API key, model và endpoint FPT

Điền ba giá trị trong cell dưới rồi chạy cell trước inference validation/test. Cấu hình này ghi đè việc đọc `.env` trong kernel hiện tại, không sửa file `.env` và không cần chạy lại bước CPU. Sau khi đổi giá trị, chạy lại cell để áp dụng. Không thay đổi cấu hình giữa lúc một cell inference đang chạy.

Xóa cell không hủy cấu hình đã nạp trong kernel; restart kernel để quay về cơ chế `.env`. Đổi model hoặc endpoint sẽ dùng cache request tương ứng.


In [ ]:
# Điền trực tiếp tại đây, sau đó chạy cell này trước inference.
FPT_API = ""  # Dán API key của bạn vào đây.
FPT_MODEL = "DeepSeek-V4-Flash"
FPT_BASE_URL = "https://token-api.fpt.ai/v1"

import sys
import grace_h1.fpt_provider as _fpt_provider
import grace_h1.inference as _fpt_inference

if not FPT_API.strip():
    raise ValueError('Điền FPT_API trong cell này rồi chạy lại.')
if not FPT_MODEL.strip() or not FPT_BASE_URL.strip():
    raise ValueError('Điền đủ FPT_MODEL và FPT_BASE_URL.')

# Chỉ ghi đè trong bộ nhớ: provider gốc vốn đọc trực tiếp file .env.
# Giữ một snapshot để sửa biến sau đó không làm đổi request đang chạy.
def _notebook_fpt_loader(_settings={
    'api_key': FPT_API.strip(),
    'model': FPT_MODEL.strip(),
    'base_url': FPT_BASE_URL.strip().rstrip('/'),
}):
    return dict(_settings)

_fpt_provider.load_fpt_settings = _notebook_fpt_loader
_fpt_inference.load_fpt_settings = _notebook_fpt_loader
# Recovery có thể đã import hàm loader trước khi bạn chạy cell này.
if 'grace_h3v2.recovery' in sys.modules:
    sys.modules['grace_h3v2.recovery'].load_fpt_settings = _notebook_fpt_loader

print('Đã áp dụng cấu hình nhập trực tiếp cho kernel hiện tại.')
print('Model:', FPT_MODEL)
print('Endpoint:', FPT_BASE_URL)
print('API key: đã nạp (ẩn)')


## Bước 1 — CPU local: classifier và chuẩn bị validation

Entry tạo base-inputs rồi dừng. Chạy notebooks/colab_cpu_prepare.ipynb local với PHASE='base'. CPU fit classifier bằng nested train OOF, chọn gate bằng MCC/F1, chuẩn bị prompts validation và ghi base-output vào bundles/. CodeT5 frozen; CPU runner không gọi LLM.


In [ ]:
base=w.base_prepare(ROOT,CFG) if not PREPARE_IN_LOCAL_NOTEBOOK else cpu_handoff('base','base_selection')
print(pd.DataFrame([{'dataset':j['dataset'],'seed':j['seed'],'base_folder':j['base_folder']} for j in base['jobs']]))


## Bước 2 — Local: GRACE validation ngoài test

Preflight và gọi các request còn thiếu tại local; exact cache được reuse. API cần mạng. Khi validation complete, Run All tạo meta-inputs local cho lượt CPU tiếp theo.

Nếu validation cũ có partial_request_errors, chuyển sang retry-blocked-validation.ipynb, chạy đến cell gộp, rồi quay lại bước này. Không cần gọi lại các response Vertex thành công.


In [ ]:
plans=w.preflight(ROOT,CFG,'validation')
print(json.dumps(plans,indent=2))
reports=w.infer(ROOT,CFG,'validation',allow_api=RUN_INFERENCE)
print(json.dumps(reports,indent=2))
if any(r['status']!='complete' for r in reports): raise RuntimeError('Validation chưa complete. Nếu đang dùng recovery trộn model, chạy notebooks/retry-blocked-validation.ipynb đến bước gộp rồi chạy lại cell này; xem reports để biết request còn lỗi.')


## Bước 3 — CPU local: meta selector/stacking và test features

Sau validation complete, entry tạo meta-inputs trong bundles/. Chạy colab_cpu_prepare.ipynb local với PHASE='meta'. CPU fit router/selector/stacking trên development, khóa winner, tính test features không nhãn, và xuất meta-output vào bundles/. Không gọi LLM ở CPU runner.

Nhánh benefit-router chỉ dùng feature trước expert; selector/stacking dùng expert output. Full-reference/stacking có thể cần100% query: benchmark mode.


In [ ]:
meta=w.meta_prepare(ROOT,CFG) if not PREPARE_IN_LOCAL_NOTEBOOK else cpu_handoff('meta','meta_selection')
print(pd.DataFrame([{'dataset':j['dataset'],'seed':j['seed'],'winner_development':j['development_winner']} for j in meta['jobs']]))


## Bước 4 — Local: test inference và đánh giá

Winner đã khóa từ development. Local preflight/test-infer reuse exact cache hoặc gọi Vertex nếu RUN_INFERENCE=True. Test scores đã tính ở CPU runner local; nhãn test chỉ mở trong evaluate.

Test mặc định dùng **8.192 byte UTF-8**, bằng validation. Bước này chỉ dựng lại prompt từ retrieval/graph cache, không fit lại model hoặc chạy Joern. Run test 24k và bảng điểm cũ được lưu riêng; response chỉ reuse khi prompt/model/spec khớp hoàn toàn. Sau khi đổi profile, chưa có bảng điểm 8k mới cho tới khi cell inference/evaluate hoàn tất.


In [ ]:
profile=test_context.prepare(ROOT,CFG,TEST_CONTEXT_LIMIT_BYTES)
print(json.dumps(profile,indent=2))
print(json.dumps(w.preflight(ROOT,CFG,'test'),indent=2))
reports=w.infer(ROOT,CFG,'test',allow_api=RUN_INFERENCE)
print(json.dumps(reports,indent=2))
if any(r['status']!='complete' for r in reports): raise RuntimeError('Test requests chưa complete; resume inference trước evaluate.')
w.predict(ROOT,CFG)
summary=w.evaluate(ROOT,CFG)
summary=test_context.annotate_summary(ROOT,CFG,summary)


In [ ]:
table=[]
for run in summary['datasets']:
    for arm,result in run['arms'].items():
        m=result['metrics']
        table.append({'dataset':run['dataset'],'seed':run['seed'],'arm':arm,'locked_winner':arm==run['locked_development_winner'],'MCC':m['mcc'],'F1':m['f1'],'precision':m['precision'],'recall':m['recall'],'specificity':m['specificity'],'invalid':m['invalid_n'],'expert_query_ratio':result['algorithmic_expert_query_ratio'],'fallback':result['fallback_n']})
display(pd.DataFrame(table))
print('Summary:',w.stage_file(ROOT,CFG,'benchmark_summary'))
print('Không chọn lại winner dựa trên bảng test. So với gate_tuned và full_grace; chi phí routing lý thuyết tách actual API/cache.')


## Chạy lại RIÊNG Devign test — DeepSeek/FPT, 8.192 byte

**Kernel:** dùng môi trường huấn luyện trên WSL: `/mnt/d/JUNIOR/LabThayTho/GRACE-H1-SecurityProjection/.venv/bin/python` (NumPy 2.5.4, SciPy 1.18.1, scikit-learn 1.9.1, joblib 1.6.0). R2 kiểm tra phiên bản trước API. Nếu IDE chưa có kernel này, mở VS Code trong WSL và chọn Python interpreter trên; không cài lại requirements-v2.txt vì các pin cũ khác phiên bản artifact.

**Chạy lần lượt 4 cell bên dưới; không dùng Run All và không cần chạy Bước 1–4 ở trên.** Các cell tự tìm thư mục dự án và nạp helper. Cần các artifact base/meta/test_scores đã có trên máy. Không fit lại model, không gọi validation hoặc ReVeal, không chọn lại winner theo test.

Lượt này gọi lại toàn bộ 2.732 mẫu Devign test (2.731 prompt khác nhau), dùng **DeepSeek-V4-Flash / FPT AI Factory**, giới hạn **8.192 byte UTF-8 cho toàn prompt**. Không lấy response của hai run cũ hoặc cache toàn dự án. API key lấy từ cấu hình đang có hoặc hộp nhập ẩn; không ghi key vào notebook/artifact.

Giữ nguyên `RERUN_SESSION` khi chạy tiếp sau gián đoạn. Mỗi request hoàn tất được lưu ngay; chạy lại cell inference chỉ gọi các request còn lỗi/invalid. Đổi tên session sẽ tạo lượt gọi mới cho toàn bộ prompt. Không chạy luồng inference khác trong cùng kernel lúc các cell này đang chạy.

Kết quả nằm riêng trong `runs/h3v2/inference_reruns/<RERUN_SESSION>/`: `expert/` có prediction, raw response, prompt, manifest; `predictions/devign/42/` có kết quả mọi arm và `evaluation.json`; `benchmark_summary.json` là bảng tổng hợp mới. Báo cáo cũ và ReVeal được giữ nguyên. Raw response vẫn là `raw_responses.jsonl` theo định dạng gốc của pipeline.

Gọi lại cùng model/prompt không bảo đảm hết hiện tượng gần như luôn đoán 0. Xem phân phối prediction sau inference và kiểm tra raw nếu vẫn bất thường; không gọi lặp/chọn run theo điểm test. Validation cũ trộn Qwen/Vertex và DeepSeek/FPT vẫn giữ nguyên, nên kết quả còn giới hạn exploratory.


In [ ]:
# R1 — Cấu hình độc lập; không cần chạy các cell trước.
from pathlib import Path
import os, sys, json, importlib.util, getpass
_candidates = [Path.cwd(), Path.cwd().parent,
    Path('D:/JUNIOR/LabThayTho/GRACE-H3-BenchmarkEnsemble'),
    Path('/mnt/d/JUNIOR/LabThayTho/GRACE-H3-BenchmarkEnsemble')]
if os.environ.get('GRACE_H3V2_ROOT'):
    _candidates.insert(0, Path(os.environ['GRACE_H3V2_ROOT']))
RERUN_ROOT = next((p.resolve() for p in _candidates if (p/'configs/benchmark_v2.json').is_file()), None)
if RERUN_ROOT is None:
    raise RuntimeError('Đặt GRACE_H3V2_ROOT trỏ tới thư mục dự án H3-B.')
if str(RERUN_ROOT) not in sys.path:
    sys.path.insert(0, str(RERUN_ROOT))
from grace_h3v2 import workflow as _rerun_workflow
from grace_h1.fpt_provider import load_fpt_settings as _rerun_load_settings
from grace_h1.io import load_local_environment as _rerun_load_env
_rerun_load_env(RERUN_ROOT)
_spec = importlib.util.spec_from_file_location('devign_test_rerun', RERUN_ROOT/'tools/rerun_devign_test.py')
devign_rerun = importlib.util.module_from_spec(_spec)
_spec.loader.exec_module(devign_rerun)
RERUN_CFG = _rerun_workflow.config(RERUN_ROOT)
print('Runtime:', devign_rerun.check_runtime(RERUN_ROOT, RERUN_CFG))
RERUN_SESSION = 'devign_fpt_8192_rerun_20261021'  # Giữ tên này để resume.
RERUN_WORKERS = 4
_existing = _rerun_load_settings() or {}
_key = globals().get('FPT_API', '') or os.environ.get('FPT_API', '') or _existing.get('api_key', '')
if not _key.strip():
    _key = getpass.getpass('FPT API key (ẩn): ')
if not _key.strip():
    raise ValueError('Cần FPT API key để inference.')
RERUN_SETTINGS = {'api_key': _key.strip(), 'model': 'DeepSeek-V4-Flash',
                  'base_url': 'https://token-api.fpt.ai/v1'}
del _key, _existing
print('ROOT:', RERUN_ROOT)
print('Session:', RERUN_SESSION, '| DeepSeek-V4-Flash / FPT AI Factory | 8192 byte')


In [ ]:
# R2 — Chỉ kiểm tra/chuẩn bị, chưa gọi API.
_prepared = devign_rerun.prepare(RERUN_ROOT, RERUN_CFG, RERUN_SESSION)
print(json.dumps(_prepared, indent=2))
_plan = devign_rerun.preflight(RERUN_ROOT, RERUN_CFG, RERUN_SETTINGS, RERUN_SESSION)
print(json.dumps(_plan, indent=2))
# Lần đầu: queries=2732, unique_prompts=2731, cached_queries=0.
# Khi resume: new_unique_requests giảm theo số request đã hoàn tất trong session này.


In [ ]:
# R3 — GỌI API Devign test. Có thể chạy lại cell này để resume cùng session.
_rerun_report = devign_rerun.infer(
    RERUN_ROOT, RERUN_CFG, RERUN_SETTINGS, RERUN_SESSION,
    workers=RERUN_WORKERS, allow_api=True)
print(json.dumps(_rerun_report, indent=2))
if _rerun_report['status'] != 'complete':
    raise RuntimeError('Còn request lỗi: xem report, xử lý nguyên nhân rồi chạy lại R3 để resume.')
print('Phân phối prediction:', _rerun_report['prediction_counts'])
if _rerun_report['near_constant_output']:
    print('CẢNH BÁO: >=99% nhãn valid cùng một lớp. Kiểm tra raw response; không chọn lại run theo điểm test.')


In [ ]:
# R4 — Dùng mô hình đã khóa để tính lại MỌI arm Devign và hai đối chứng.
# Không gọi API, không fit lại model, không chọn lại winner.
import pandas as pd
_rerun_summary = devign_rerun.evaluate(RERUN_ROOT, RERUN_CFG, RERUN_SETTINGS, RERUN_SESSION)
_result_rows = []
for _dataset in _rerun_summary['datasets']:
    _all = {name: result['metrics'] for name, result in _dataset['arms'].items()}
    _all.update(_dataset['trivial'])
    for _name, _m in _all.items():
        _result_rows.append({'arm': _name,
            'locked_winner': _name == _dataset['locked_development_winner'],
            'N': _m['n'], 'valid': _m['valid_n'], 'invalid': _m['invalid_n'],
            'Accuracy (%)': 100*_m['accuracy_valid_only'],
            'P (%)': 100*_m['precision'], 'R (%)': 100*_m['recall'],
            'F1 (%)': 100*_m['f1'], 'MCC': _m['mcc'],
            'Balanced accuracy (%)': 100*_m['balanced_accuracy'],
            'confusion matrix': _m['confusion_matrix_valid_only']})
display(pd.DataFrame(_result_rows))
print('Kết quả:', RERUN_ROOT/'runs/h3v2/inference_reruns'/RERUN_SESSION/'benchmark_summary.json')
print(_rerun_summary['limitations'])
